In [1]:
import random
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from omegaconf import OmegaConf
from rdkit import Chem
from rdkit.Chem import  DataStructs, rdMolDescriptors, Descriptors
import torch_geometric.data as gd
from gflownet.proxy.mol_utils import smiles2graph
from gflownet.utils.conditioning import TemperatureConditional
from gflownet.models.graph_transformer import GraphTransformerGFN
from gflownet.algo.trajectory_balance import TrajectoryBalance
from gflownet.algo.flow_matching import FlowMatching
from gflownet.envs.graph_building_env import GraphBuildingEnv
from gflownet.envs.frag_mol_env import FragMolBuildingEnvContext
from gflownet.models import bengio2021flow
from gflownet.proxy.model import load_proxy_to_gflow
import warnings
from scipy.stats import pearsonr, spearmanr
from rdkit import RDLogger   

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [5]:
id = "example"
current_dir = Path.cwd()
proxy_model = load_proxy_to_gflow("src/gflownet/proxy/model_params.txt","src/gflownet/proxy/best_model.pt")
# Iterate over each algo
yaml_dir = current_dir / "src" / "gflownet" / "tasks" / "logs" / id / "config.yaml"
model_dir = current_dir / "src" / "gflownet" / "tasks" / "logs" / id / "model_state.pt"
cfg = OmegaConf.load(yaml_dir)
# Load env
env = GraphBuildingEnv()
temp_cond = TemperatureConditional(cfg)
num_cond_dim = temp_cond.encoding_size()
ctx = FragMolBuildingEnvContext(
    max_frags=cfg.algo.max_nodes,
    num_cond_dim=num_cond_dim,
    fragments=bengio2021flow.FRAGMENTS,
)
# Load GFN Model
model = GraphTransformerGFN(
    env_ctx=ctx,
    cfg=cfg,
    num_graph_out=cfg.algo.tb.do_predict_n + 1,
    do_bck=cfg.algo.tb.do_parameterize_p_b,
)
model.load_state_dict((torch.load(model_dir)["models_state_dict"][0])) 
model.eval()
# Load Algo
# Load cond_info
algo = TrajectoryBalance(env, ctx, cfg)
seed = 42
total_smiles = []
total_preds = []
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
num_gen = 50
for _ in range(5):
    # Sample from the GFlowNet
    cond_info = temp_cond.sample(num_gen)["encoding"]
    samples = algo.create_training_data_from_own_samples(model=model, n=num_gen, cond_info=cond_info)
    trajectories = [sample["traj"] for sample in samples]
    rdkit_mols = [ctx.graph_to_obj(traj[-1][0]) for traj in trajectories]
    smiles = [Chem.MolToSmiles(mol) for mol in rdkit_mols]
    # Get preds
    graphs = [smiles2graph(Chem.MolFromSmiles(smile)) for smile in smiles]
    batch = gd.Batch.from_data_list([g for g in graphs if g is not None])
    preds = (
        proxy_model(batch.x, batch.edge_index, batch.edge_attr, batch.batch).squeeze(dim=-1).cpu().detach().numpy()
    )
    min_sol = -13.71
    max_sol = 2.41
    # scaled_preds = list(1 - ((preds - min_sol) / (max_sol - min_sol)))
    total_smiles.extend(smiles)
    total_preds.extend(preds)
# Add each seed smiles and preds to total list
df_all = pd.DataFrame({"smiles": total_smiles, "reward": total_preds})

In [6]:
df_all

,smiles,reward
0,O=S(=O)([O-])c1c(-c2cc(C3=CNC=CC3)c3ccccc3c2)n...,-6.398878
1,CC(=O)NC1CC(N2C=CCC(n3c4nc(=O)[nH]c(=O)c-4nc4c...,-4.355134
2,CCc1cc2nc3c(=O)[nH]c(=O)nc-3n(C3CC(C4CCCCC4)NC...,-7.511293
3,O=[PH]([O-])C1NCCC1C1CC(c2ccc3ccccc3c2)N(c2c[n...,-3.571855
4,C[SH+]n1ccc(N2CCN(C3CCC(c4c[nH]c(-c5ncnc6[nH]c...,-2.982661
...,...,...
245,O=[N+]([O-])c1ccnn1-c1cncnc1-c1ncnc(C2CCCCC2)c...,-6.276674
246,CC(C)(O)[n+]1cccc(C2CNc3nc(P(=O)([O-])O)[nH]c(...,-0.983867
247,N=C(N)OCC1CNc2nc(-c3nc([SH](=O)=O)cs3)[nH]c(=O...,-3.041484
248,O=c1[nH]c(-c2cccs2)nc2c1NC(c1c[nH]c(-c3cccc(-c...,-7.793057
